# Level 3 — AI Challenge

Welcome to Level 3.

This is a radiotherapz AI challenge, buildig on the [https://aapm.onlinelibrary.wiley.com/doi/epdf/10.1002/mp.14845](openKBP) challenge and dataset. 

The aim is to predict photon radiotherapy treatment dose from the patient CT and syructure set. 

You are given a **working baseline**, training data, validation data, and an evaluation pipeline.

Your task is to understand the supplied system and improve it.

There is no single intended solution.

The challenge is deliberately designed so that improvements can come from many places:

```text
data representation
sampling
augmentation
model architecture
loss function
optimization
training strategy
inference
```

A more complicated solution is not automatically a better solution.

The final result will be evaluated on data that are not available during model development.

## Challenge rules

Please read these before starting.

### You may

- modify the supplied training and inference code;
- change the input representation;
- change patch sampling and augmentation;
- change the network architecture;
- change the loss function;
- change the optimizer and training strategy;
- use PyTorch, MONAI, NumPy, and other normal Python libraries;
- consult documentation, papers, tutorials, and general AI resources;
- use LLMs and coding assistants.

### LLMs are allowed

Using an LLM does not remove your responsibility for the result.

You should be able to explain the important parts of your final solution:

```text
What did you change?
Why did you expect it to help?
What happened on validation?
What did it cost computationally?
```

Do not treat an LLM agreeing with you as evidence that an approach is scientifically correct.

### You may not

- obtain additional patient cases beyond the data supplied for this challenge;
- download or reconstruct the full source dataset to gain access to additional patients;
- search for or use hidden test reference values;
- copy an existing solution to this challenge or a closely corresponding external competition solution;
- use pretrained weights specifically trained for this dose-prediction task;
- use external patient datasets for training;
- modify the official evaluation code in order to change your score.

The challenge relies on an **honor system**.

Technically, some of these things may be possible. That does not make them part of the challenge.

The point is to find out what *you* can improve using the same information available to everyone else.

## Honor code

By continuing, you agree to work only with the patient data supplied for the course and to respect the rules above.

LLMs are allowed. External challenge data and hidden reference data are not.

Set the value below to `True` before continuing.

In [ ]:
I_AGREE_TO_HONOR_CODE = False

assert I_AGREE_TO_HONOR_CODE, (
    "Please read the challenge rules and set "
    "I_AGREE_TO_HONOR_CODE = True before continuing."
)

## 1. Start a Colab GPU runtime

In Colab select:

**Runtime → Change runtime type → GPU**

The baseline also runs on CPU, but a GPU will make experimentation faster.

The challenge is intended to remain feasible on a standard free Colab runtime. Paid compute is not required.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "No GPU detected. The code may still run on CPU, "
        "but consider switching the Colab runtime to GPU."
    )

## 2. Upload the supplied challenge folder

Copy the challenge folder from the USB key to your computer and upload it to the Colab session using the **Files** panel on the left.

The folder should contain approximately:

```text
challenge/
├── main.py
├── provided_code/
│   ├── monai_data.py
│   ├── network_architectures_MONAI.py
│   ├── network_functions_MONAI.py
│   ├── dose_evaluation_npz.py
│   └── utils.py
└── provided-data/
    ├── train-pats/
    │   └── ...
    └── validation-pats/
        └── ...
```

The patient data are already prepared. You do not need to convert them.

If your browser uploads the supplied folder as an archive, extract it under `/content` before continuing.

In [ ]:
from pathlib import Path

def find_challenge_root(search_root=Path("/content")):
    candidates = []

    for main_file in search_root.rglob("main.py"):
        project_root = main_file.parent

        if (
            (project_root / "provided_code").is_dir()
            and (project_root / "provided-data").is_dir()
        ):
            candidates.append(project_root)

    if not candidates:
        raise FileNotFoundError(
            "Could not find the challenge folder below /content.\n"
            "Upload the supplied folder, then run this cell again."
        )

    if len(candidates) > 1:
        print("Multiple possible challenge folders found:")
        for candidate in candidates:
            print(" -", candidate)
        print("\nUsing the first one.")

    return candidates[0]


PROJECT_ROOT = find_challenge_root()

print("Challenge folder:")
print(PROJECT_ROOT)

In [ ]:
%cd $PROJECT_ROOT

## 3. Install the required environment

The challenge uses PyTorch and MONAI.

Colab already provides PyTorch. The cell below installs the MONAI version used for the baseline.

In [ ]:
!pip -q install "monai==1.6.1"

## 4. Check the supplied data and code

Before changing anything, make sure the baseline project is complete.

Do not start by modifying the code.

First establish that the supplied version runs.

In [ ]:
from pathlib import Path

required = [
    Path("main.py"),
    Path("provided_code/monai_data.py"),
    Path("provided_code/network_architectures_MONAI.py"),
    Path("provided_code/network_functions_MONAI.py"),
    Path("provided_code/dose_evaluation_npz.py"),
    Path("provided_code/utils.py"),
    Path("provided-data/train-pats"),
    Path("provided-data/validation-pats"),
]

missing = [str(path) for path in required if not path.exists()]

if missing:
    raise FileNotFoundError(
        "The challenge folder appears incomplete:\n"
        + "\n".join(missing)
    )

train_npz = list(Path("provided-data/train-pats").rglob("patient.npz"))
validation_npz = list(Path("provided-data/validation-pats").rglob("patient.npz"))

print("Training cases:", len(train_npz))
print("Validation cases:", len(validation_npz))
print("Challenge files found.")

## 5. Understand the baseline before changing it

The project is intentionally small.

The main places to look are:

### `main.py`

Controls the overall experiment:

```text
load data
→ train
→ predict
→ evaluate
```

### `provided_code/monai_data.py`

Controls what the network sees and how training samples are produced.

Possible questions:

```text
What are the input channels?
How is CT intensity represented?
How are patches sampled?
Would augmentation make sense?
```

### `provided_code/network_architectures_MONAI.py`

Contains the baseline network.

Possible questions:

```text
Is the network too small?
Would a different architecture help?
Does extra model capacity actually improve validation?
```

### `provided_code/network_functions_MONAI.py`

Contains the training loop, loss, optimizer, checkpointing, and inference.

Possible questions:

```text
Is the objective appropriate?
Could the optimizer/training schedule be improved?
Are the inference settings important?
```

### `provided_code/dose_evaluation_npz.py`

Contains the official validation metrics.

**Do not change the official evaluator to improve your score.**

## 6. Run the baseline

Open `main.py` in the Colab file browser before running it.

For your first run, use the supplied settings or a short dry run.

The purpose of the first run is to establish:

```text
Does the pipeline work?
How long does training take?
What are the baseline validation scores?
What does the predicted dose look like?
```

Run the project with:

In [ ]:
!python main.py

## 7. Record the baseline

Before you change anything, write down the baseline.

This matters. Otherwise you will eventually have a more complicated model without knowing whether it is actually better.

Use something like:

```text
Experiment: baseline
Change: none
Training:
Validation Dose score:
Validation DVH score:
Runtime:
Notes:
```

## 8. Improve one thing at a time

Do not ask only:

> How can I make the model better?

Use a concrete hypothesis.

For example:

```text
Current state:
One random training patch is sampled per patient.

Proposed change:
Sample more than one patch.

Reason:
The model will see greater spatial diversity during one epoch.

What stays unchanged:
Architecture, loss, optimizer, validation procedure.

Evidence:
Dose score and DVH score on the validation cohort.
```

Then run the experiment.

A useful challenge workflow is:

```text
baseline
↓
one hypothesis
↓
change code
↓
train
↓
validate
↓
inspect numerical + visual results
↓
keep or reject the change
```

Do not keep a change only because the code runs.

## 9. Keep a small experiment log

You do not need a sophisticated experiment-management system.

A simple table is enough.

| Experiment | Change | Why? | Dose score | DVH score | Keep? |
|---|---|---|---:|---:|---|
| Baseline | None | Reference |  |  |  |
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |

Try to avoid changing five things at once. If the score improves, you should have some idea why.

## 10. Validation is not the final test

You may use the supplied validation cohort repeatedly while developing your model.

This means that, over time, you can also overfit your *decisions* to the validation set.

The final ranking therefore uses a separate test cohort.

The test reference values are retained by the instructor.

When the final test inputs are made available, use your already-selected method. Do not start another round of model selection using the test set.

## 11. Final submission

At the end of the challenge, you will be told how the final test inputs are provided.

Your final pipeline should generate a submission archive using the supplied project structure.

Submit only the requested prediction archive.

Before submission, be prepared to explain briefly:

```text
1. What did you change relative to the baseline?
2. Why?
3. Which change helped most on validation?
4. Did anything that sounded sensible fail?
5. How did you use an LLM, if at all?
```

There is no penalty for using an LLM.

There is a penalty for not understanding what your final code does.

# Final reminder

The goal is not to produce the longest code or the largest model.

The goal is to make a defensible improvement to a working AI pipeline.

```text
Change something for a reason.
Measure what happened.
Keep evidence.
```

Good luck.